<a href="https://colab.research.google.com/github/michalis0/DataScience_and_MachineLearning/blob/master/Labs/08-gen-ai/generative-ai.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Lab 8 · Generative AI

## Learning goals

- Build a reusable prompt and compare free-form and structured responses.
- Summarize a source passage and classify its sentiment.
- Use vector similarity to retrieve text relevant to a query.

## Before you start

Run the local examples in order. The optional live examples require packages and an API key configured in your environment.

For a new environment, install the [lab requirements](../requirements.txt) with `python -m pip install -r Labs/requirements.txt` from the repository root.

## 1. Build a prompt

Separate the task instructions from the input. Compare the wording of a free-form prompt with a prompt that asks for a specific response format.

In [ ]:
from string import Template

summary_prompt = Template(
    "Summarize the passage in one sentence. Use only facts from the passage.\n\n$passage"
)
passage = (
    "An information system combines people, processes, and technology. "
    "An inventory system records stock movements so staff can decide when to reorder. "
    "Reliable records help managers compare what is available with what customers need."
)
print(summary_prompt.substitute(passage=passage))

## 2. Summarize and label text

Run these reference examples first. The summary is an extractive baseline. The sentiment example is a small labeled dataset that you can use to check an optional model response.

In [ ]:
import pandas as pd

extractive_summary = passage.split(". ", 1)[0] + "."
sentiment_examples = pd.DataFrame(
    [
        ("I love how easy this is to use.", "positive"),
        ("The delivery arrived two days late.", "negative"),
        ("The box contains two cables.", "neutral"),
    ],
    columns=["text", "expected_sentiment"],
)
print(extractive_summary)
display(sentiment_examples)

## 3. Retrieve relevant context

Use TF-IDF and cosine similarity as a local baseline before comparing semantic embeddings. The result is a source passage you can include in a prompt.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

documents = [
    "Inventory management tracks stock movements and reorder levels.",
    "Information systems combine people, processes, and technology.",
    "A recommender system uses interaction data to rank relevant items.",
]
question = "How do you track inventory stock?"
vectorizer = TfidfVectorizer(stop_words="english")
document_vectors = vectorizer.fit_transform(documents)
query_vector = vectorizer.transform([question])
scores = cosine_similarity(query_vector, document_vectors)[0]
context = documents[scores.argmax()]
print(context)

## 4. Optional live comparison

Install `langchain-openai` and configure `OPENAI_API_KEY`, `OPENAI_CHAT_MODEL`, and `OPENAI_EMBEDDING_MODEL` in your environment. See the [API key guide](./Guide-API-Key-OpenAI.ipynb). Set `RUN_LIVE` to `True` to send requests. These calls may incur API charges. Record the token usage returned by the chat model and use the pricing for your configured model to estimate cost.

In [ ]:
import os
from typing import Literal

RUN_LIVE = False

if RUN_LIVE:
    from langchain_openai import ChatOpenAI, OpenAIEmbeddings
    from pydantic import BaseModel

    class SentimentResult(BaseModel):
        sentiment: Literal["positive", "neutral", "negative"]

    chat = ChatOpenAI(model=os.environ["OPENAI_CHAT_MODEL"], temperature=0)
    summary = chat.invoke(summary_prompt.substitute(passage=passage))
    print(summary.content)
    print("Token usage:", summary.usage_metadata)

    classify = chat.with_structured_output(SentimentResult)
    for row in sentiment_examples.itertuples(index=False):
        result = classify.invoke(f"Label the sentiment of this text: {row.text}")
        print(result.sentiment, row.expected_sentiment)

    embeddings = OpenAIEmbeddings(model=os.environ["OPENAI_EMBEDDING_MODEL"])
    semantic_scores = cosine_similarity(
        [embeddings.embed_query(question)], embeddings.embed_documents(documents)
    )[0]
    print(documents[semantic_scores.argmax()])
else:
    print("Live model calls skipped.")

## Practice

1. Rewrite the summary prompt to specify an audience and compare the resulting summary with the baseline.
2. Add a sentence with mixed sentiment. Decide on a label and explain which words guided you.
3. Ask the same retrieval question with different vocabulary. Check whether the best-ranked passage changes.

In [ ]:
# Write your solution here.